# Oil Market Explorer
**Source:** EI Statistical Review of World Energy 2025 → SQLite  
**Database:** `oil_market.db`

This notebook connects to the SQLite database built from the EI Stats Review Excel file. Every chart and table is driven by SQL — change the query, change the output.

---

## Setup

In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

DB_PATH = "oil_market.db"
conn = sqlite3.connect(DB_PATH)

import plotly.io as pio
pio.templates.default = "plotly_dark"

LAYOUT = dict(
    font_family="DM Sans, system-ui, sans-serif",
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(15,15,20,1)",
    margin=dict(l=50, r=30, t=50, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    hovermode="x unified",
)

def style(fig, **kwargs):
    fig.update_layout(**LAYOUT, **kwargs)
    fig.update_xaxes(gridcolor="rgba(255,255,255,0.05)", zeroline=False)
    fig.update_yaxes(gridcolor="rgba(255,255,255,0.05)", zeroline=False)
    return fig

def q(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print("Connected to", DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

Connected to oil_market.db
Tables: ['oil_production', 'oil_consumption', 'refinery_throughput', 'refinery_capacity', 'spot_crude_prices', 'historical_oil_prices', 'refining_margins']


## Database schema reference

In [2]:
for table in q("SELECT name FROM sqlite_master WHERE type='table'")["name"]:
    info = q(f"PRAGMA table_info({table})")
    count = q(f"SELECT COUNT(*) as n FROM {table}")["n"][0]
    years = q(f"SELECT MIN(year) as lo, MAX(year) as hi FROM {table}")
    cols_str = ", ".join(f"{r['name']} ({r['type']})" for _, r in info.iterrows())
    print(f"  {table}  |  {count:,} rows  |  {years['lo'][0]}\u2013{years['hi'][0]}  |  {cols_str}")

  oil_production  |  3,743 rows  |  1965–2024  |  entity (TEXT), year (INT), value_kbd (REAL)
  oil_consumption  |  5,817 rows  |  1965–2024  |  entity (TEXT), year (INT), value_kbd (REAL)
  refinery_throughput  |  3,876 rows  |  1980–2024  |  entity (TEXT), year (INT), value_kbd (REAL)
  refinery_capacity  |  5,120 rows  |  1965–2024  |  entity (TEXT), year (INT), value_kbd (REAL)
  spot_crude_prices  |  200 rows  |  1972–2024  |  benchmark (TEXT), year (INT), price_usd (REAL)
  historical_oil_prices  |  164 rows  |  1861–2024  |  year (INT), nominal_usd (REAL), real_2024_usd (REAL)
  refining_margins  |  75 rows  |  2000–2024  |  region (TEXT), year (INT), margin_usd_bbl (REAL)


## Entity lookup

Find what countries/regions are available. Change `table` to explore different datasets.

In [3]:
table = "oil_production"  # try: oil_consumption, refinery_capacity, refinery_throughput

entities = q(f"SELECT DISTINCT entity FROM {table} ORDER BY entity")
print(f"{len(entities)} entities in {table}:\n")
for e in entities["entity"]:
    print(f"  {e}")

69 entities in oil_production:

  Algeria
  Angola
  Argentina
  Australia
  Azerbaijan
  Brazil
  Brunei
  Canada
  Chad
  China
  Colombia
  Denmark
  Ecuador
  Egypt
  Equatorial Guinea
  European Union #
  Gabon
  Guyana
  India
  Indonesia
  Iran
  Iraq
  Italy
  Kazakhstan
  Kuwait
  Libya
  Malaysia
  Mexico
  Nigeria
  Non-OECD
  Non-OPEC
  Norway
  OPEC
  Oman
  Other Africa
  Other Asia Pacific
  Other CIS
  Other Europe
  Other Middle East
  Other S. & Cent. America
  Peru
  Qatar
  Republic of Congo
  Romania
  Russian Federation
  Saudi Arabia
  South Sudan
  Sudan
  Syria
  Thailand
  Total Africa
  Total Asia Pacific
  Total CIS
  Total Europe
  Total Middle East
  Total North America
  Total S. & Cent. America
  Total World
  Trinidad & Tobago
  Tunisia
  Turkmenistan
  US
  United Arab Emirates
  United Kingdom
  Uzbekistan
  Venezuela
  Vietnam
  Yemen
  of which: OECD


---
## 1. Global supply / demand balance

The bar shows the implied surplus (production - consumption). Positive = oversupplied, negative = stock draw.

In [4]:
df = q('''
    SELECT p.year,
           p.value_kbd AS production,
           c.value_kbd AS consumption,
           p.value_kbd - c.value_kbd AS surplus
    FROM oil_production p
    JOIN oil_consumption c ON p.year = c.year AND p.entity = c.entity
    WHERE p.entity = 'Total World' AND p.year >= 1965
    ORDER BY p.year
''')

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.year, y=df.production, name="Production",
              line=dict(color="#2b8a3e", width=2)))
fig.add_trace(go.Scatter(x=df.year, y=df.consumption, name="Consumption",
              line=dict(color="#e8590c", width=2)))
fig.add_trace(go.Bar(x=df.year, y=df.surplus, name="Surplus / Deficit",
              marker_color=df.surplus.apply(lambda v: "#364fc7" if v >= 0 else "#c92a2a"),
              opacity=0.5))
style(fig, title="Global Oil Supply vs Demand (kbd)", yaxis_title="Thousand barrels / day")
fig.show()

## 2. OPEC vs Non-OPEC production

In [5]:
df = q('''
    SELECT year, entity, value_kbd
    FROM oil_production
    WHERE entity IN ('OPEC', 'Non-OPEC') AND year >= 1990
    ORDER BY year
''')

fig = px.area(df, x="year", y="value_kbd", color="entity",
              color_discrete_map={"OPEC": "#e8590c", "Non-OPEC": "#364fc7"},
              labels={"value_kbd": "kbd", "year": ""})
style(fig, title="OPEC vs Non-OPEC Production (kbd)")
fig.show()

## 3. Top producers — customisable

Change `COUNTRIES` and `START_YEAR` to plot any combination.

In [6]:
# ── CHANGE THESE ──
COUNTRIES = ["US", "Saudi Arabia", "Russian Federation", "Canada", "China", "Brazil"]
START_YEAR = 2000

placeholders = ",".join(["?"] * len(COUNTRIES))
df = q(f'''
    SELECT entity, year, value_kbd
    FROM oil_production
    WHERE entity IN ({placeholders}) AND year >= ?
    ORDER BY year
''', COUNTRIES + [START_YEAR])

fig = px.line(df, x="year", y="value_kbd", color="entity",
              labels={"value_kbd": "kbd", "year": "", "entity": ""},
              color_discrete_sequence=px.colors.qualitative.Bold)
style(fig, title=f"Oil Production \u2014 Selected Countries ({START_YEAR}\u20132024)")
fig.show()

## 4. Production ranking table — 2024

In [7]:
df = q('''
    SELECT entity,
           value_kbd,
           ROUND(value_kbd / (SELECT value_kbd FROM oil_production
                              WHERE entity='Total World' AND year=2024) * 100, 1) AS pct_world
    FROM oil_production
    WHERE year = 2024
      AND entity NOT LIKE 'Total%'
      AND entity NOT LIKE 'of which%'
      AND entity NOT LIKE 'Non-%'
      AND entity NOT LIKE 'OPEC%'
      AND entity NOT LIKE 'European%'
      AND entity NOT LIKE 'Other%'
    ORDER BY value_kbd DESC
    LIMIT 20
''')
df.columns = ["Country", "Production (kbd)", "% World"]
df.index = range(1, len(df) + 1)
df["Production (kbd)"] = df["Production (kbd)"].map(lambda v: f"{v:,.0f}")
display(df)

,Country,Production (kbd),% World
1,US,"20,135",20.8
2,Saudi Arabia,"10,856",11.2
3,Russian Federation,"10,752",11.1
4,Canada,"5,888",6.1
5,Iran,"5,062",5.2
6,Iraq,"4,398",4.5
7,China,"4,264",4.4
8,United Arab Emirates,"4,006",4.1
9,Brazil,"3,466",3.6
10,Kuwait,"2,719",2.8


## 5. Top consumers — customisable

In [8]:
# ── CHANGE THESE ──
COUNTRIES = ["US", "China", "India", "Japan", "Saudi Arabia", "Russian Federation"]
START_YEAR = 1990

placeholders = ",".join(["?"] * len(COUNTRIES))
df = q(f'''
    SELECT entity, year, value_kbd
    FROM oil_consumption
    WHERE entity IN ({placeholders}) AND year >= ?
    ORDER BY year
''', COUNTRIES + [START_YEAR])

fig = px.line(df, x="year", y="value_kbd", color="entity",
              labels={"value_kbd": "kbd", "year": "", "entity": ""},
              color_discrete_sequence=px.colors.qualitative.Vivid)
style(fig, title=f"Oil Consumption \u2014 Selected Countries ({START_YEAR}\u20132024)")
fig.show()

## 6. Regional consumption — stacked area

Shows the demand shift from OECD (North America + Europe) toward Asia Pacific.

In [9]:
df = q('''
    SELECT entity, year, value_kbd
    FROM oil_consumption
    WHERE entity LIKE 'Total%' AND entity != 'Total World' AND year >= 1990
    ORDER BY year
''')

fig = px.area(df, x="year", y="value_kbd", color="entity",
              labels={"value_kbd": "kbd", "year": "", "entity": ""},
              color_discrete_sequence=px.colors.qualitative.Bold)
style(fig, title="Regional Oil Consumption (kbd) \u2014 Stacked")
fig.show()

## 7a. Long-run crude prices since 1900 — nominal vs real (2024$)

In [10]:
df = q("SELECT * FROM historical_oil_prices WHERE year >= 1900 ORDER BY year")

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.year, y=df.real_2024_usd, name="Real (2024$)", fill="tozeroy",
              line=dict(color="#e8590c"), fillcolor="rgba(232,89,12,0.2)"))
fig.add_trace(go.Scatter(x=df.year, y=df.nominal_usd, name="Nominal",
              line=dict(color="#364fc7", width=1.5)))
style(fig, title="Crude Oil Prices Since 1900 ($/bbl)", yaxis_title="$/barrel")
fig.show()

## 7b. Spot benchmarks — Brent, WTI, Dubai

In [11]:
df = q("SELECT * FROM spot_crude_prices ORDER BY year")

fig = px.line(df, x="year", y="price_usd", color="benchmark",
              labels={"price_usd": "$/bbl", "year": "", "benchmark": ""},
              color_discrete_map={"Brent": "#e8590c", "WTI": "#364fc7",
                                  "Dubai": "#087f5b", "Nigerian Forcados": "#e67700"})
style(fig, title="Spot Crude Benchmarks \u2014 Annual Average ($/bbl)")
fig.show()

## 8. Benchmark spreads — Brent-WTI and Brent-Dubai

A wide Brent-WTI spread historically incentivised US imports; the flip to negative post-shale reflected US oversupply. Brent-Dubai reflects the Atlantic vs Pacific basin differential.

In [12]:
df = q('''
    SELECT b.year,
           b.price_usd - w.price_usd AS brent_wti,
           b.price_usd - d.price_usd AS brent_dubai
    FROM spot_crude_prices b
    JOIN spot_crude_prices w ON b.year = w.year AND w.benchmark = 'WTI'
    JOIN spot_crude_prices d ON b.year = d.year AND d.benchmark = 'Dubai'
    WHERE b.benchmark = 'Brent'
    ORDER BY b.year
''')

fig = go.Figure()
fig.add_trace(go.Bar(x=df.year, y=df.brent_wti, name="Brent \u2013 WTI",
              marker_color=df.brent_wti.apply(lambda v: "#2b8a3e" if v >= 0 else "#c92a2a")))
fig.add_trace(go.Scatter(x=df.year, y=df.brent_dubai, name="Brent \u2013 Dubai",
              line=dict(color="#e67700", width=2)))
fig.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)")
style(fig, title="Crude Benchmark Spreads ($/bbl)", yaxis_title="$/barrel")
fig.show()

## 9. Refinery capacity, throughput & utilisation

Utilisation = throughput / capacity. Above ~85% refiners have pricing power on product margins. Below ~80% the market is long on capacity.

In [13]:
df = q('''
    SELECT c.year,
           c.value_kbd AS capacity,
           t.value_kbd AS throughput,
           ROUND(t.value_kbd / c.value_kbd * 100, 1) AS utilisation_pct
    FROM refinery_capacity c
    JOIN refinery_throughput t ON c.year = t.year AND c.entity = t.entity
    WHERE c.entity = 'Total World' AND c.year >= 1980
    ORDER BY c.year
''')

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df.year, y=df.capacity, name="Capacity", fill="tozeroy",
              line=dict(color="#364fc7"), fillcolor="rgba(54,79,199,0.15)"), secondary_y=False)
fig.add_trace(go.Scatter(x=df.year, y=df.throughput, name="Throughput",
              line=dict(color="#e8590c", width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=df.year, y=df.utilisation_pct, name="Utilisation %",
              line=dict(color="#2b8a3e", width=2, dash="dot")), secondary_y=True)
style(fig, title="Global Refinery Capacity vs Throughput (kbd)")
fig.update_yaxes(title_text="kbd", secondary_y=False)
fig.update_yaxes(title_text="Utilisation %", secondary_y=True, range=[60, 100])
fig.show()

## 10. Regional refining margins

Benchmark gross margins for three representative configurations.

In [14]:
df = q("SELECT * FROM refining_margins ORDER BY year")

fig = px.bar(df, x="year", y="margin_usd_bbl", color="region", barmode="group",
             color_discrete_map={
                 "USGC Medium Sour Coking": "#e8590c",
                 "NWE Light Sweet Cracking": "#364fc7",
                 "Singapore Medium Sour Hydrocracking": "#087f5b"},
             labels={"margin_usd_bbl": "$/bbl", "year": "", "region": ""})
style(fig, title="Regional Refining Margins ($/bbl)")
fig.show()

## 11. Country deep-dive — production vs consumption

Change `COUNTRY` to any entity. The bar shows implied net imports (consumption - production).

In [23]:
# ── CHANGE THIS ──
COUNTRY = "US"

df = q('''
    SELECT p.year,
           p.value_kbd AS production,
           c.value_kbd AS consumption,
           c.value_kbd - p.value_kbd AS net_imports
    FROM oil_production p
    JOIN oil_consumption c ON p.year = c.year AND p.entity = c.entity
    WHERE p.entity = ?
    ORDER BY p.year
''', [COUNTRY])

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.year, y=df.production, name="Production",
              line=dict(color="#2b8a3e", width=2)))
fig.add_trace(go.Scatter(x=df.year, y=df.consumption, name="Consumption",
              line=dict(color="#e8590c", width=2)))
fig.add_trace(go.Bar(x=df.year, y=df.net_imports, name="Net Imports (implied)",
              marker_color=df.net_imports.apply(lambda v: "#c92a2a" if v > 0 else "#364fc7"),
              opacity=0.4))
style(fig, title=f"{COUNTRY} \u2014 Production vs Consumption (kbd)")
fig.show()

## 12. Year-over-year changes

Spot demand shocks (2008-09, 2020) or supply disruptions. Change `ENTITY` and `TABLE`.

In [16]:
# ── CHANGE THESE ──
ENTITY = "Total World"
TABLE = "oil_consumption"  # or oil_production

df = q(f'''
    SELECT year, value_kbd,
           value_kbd - LAG(value_kbd) OVER (ORDER BY year) AS yoy_change,
           ROUND((value_kbd - LAG(value_kbd) OVER (ORDER BY year))
                 / LAG(value_kbd) OVER (ORDER BY year) * 100, 2) AS yoy_pct
    FROM {TABLE}
    WHERE entity = ?
    ORDER BY year
''', [ENTITY])
df = df.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(x=df.year, y=df.yoy_change, name="YoY Change (kbd)",
              marker_color=df.yoy_change.apply(lambda v: "#2b8a3e" if v >= 0 else "#c92a2a")))
style(fig, title=f"{ENTITY} \u2014 YoY Change in {'Consumption' if 'consumption' in TABLE else 'Production'} (kbd)")
fig.show()

display(Markdown("**Largest annual swings:**"))
biggest = pd.concat([df.nlargest(5, "yoy_change"), df.nsmallest(5, "yoy_change")])
biggest = biggest.sort_values("yoy_change", ascending=False).reset_index(drop=True)
biggest.columns = ["Year", "Level (kbd)", "YoY Change (kbd)", "YoY %"]
display(biggest)

**Largest annual swings:**

,Year,Level (kbd),YoY Change (kbd),YoY %
0,2021,95313.717,5196.121,5.77
1,1973,56081.497,4202.511,8.10
2,1970,45696.786,3729.657,8.89
3,1972,51878.986,3696.537,7.67
4,1969,41967.129,3438.491,8.92
5,2009,84047.690,-1352.118,-1.58
6,1982,58067.757,-1538.503,-2.58
7,1981,59606.260,-1827.353,-2.97
8,1980,61433.613,-2900.879,-4.51
9,2020,90117.596,-8780.575,-8.88


## 13. Custom SQL sandbox

Write any query. Result comes back as a DataFrame.

In [17]:
sql = '''
    SELECT p.entity,
           ROUND(p.value_kbd, 0) AS prod_2024,
           ROUND(c.value_kbd, 0) AS cons_2024,
           ROUND(c.value_kbd - p.value_kbd, 0) AS net_imports,
           ROUND(p.value_kbd / (SELECT value_kbd FROM oil_production
                                WHERE entity='Total World' AND year=2024) * 100, 1) AS prod_share_pct
    FROM oil_production p
    JOIN oil_consumption c ON p.entity = c.entity AND p.year = c.year
    WHERE p.year = 2024
      AND p.entity NOT LIKE 'Total%'
      AND p.entity NOT LIKE 'of which%'
      AND p.entity NOT LIKE 'Non-%'
      AND p.entity NOT LIKE 'OPEC%'
      AND p.entity NOT LIKE 'European%'
      AND p.entity NOT LIKE 'Other%'
    ORDER BY net_imports DESC
    LIMIT 15
'''

result = q(sql)
display(result)

,entity,prod_2024,cons_2024,net_imports,prod_share_pct
0,China,4264.0,16374.0,12110.0,4.4
1,India,735.0,5621.0,4886.0,0.8
2,Italy,93.0,1262.0,1169.0,0.1
3,Indonesia,613.0,1633.0,1020.0,0.6
4,Thailand,353.0,1253.0,900.0,0.4
5,Australia,365.0,1087.0,722.0,0.4
6,United Kingdom,653.0,1325.0,672.0,0.7
7,Vietnam,177.0,707.0,530.0,0.2
8,Malaysia,535.0,934.0,399.0,0.6
9,Romania,60.0,242.0,182.0,0.1


---
## Appendix: Full entity lists

In [18]:
for table in ["oil_production", "oil_consumption", "refinery_throughput", "refinery_capacity"]:
    entities = q(f"SELECT DISTINCT entity FROM {table} ORDER BY entity")["entity"].tolist()
    print(f"\n{table} ({len(entities)} entities):")
    for i in range(0, len(entities), 4):
        print("  " + "  |  ".join(entities[i:i+4]))


oil_production (69 entities):
  Algeria  |  Angola  |  Argentina  |  Australia
  Azerbaijan  |  Brazil  |  Brunei  |  Canada
  Chad  |  China  |  Colombia  |  Denmark
  Ecuador  |  Egypt  |  Equatorial Guinea  |  European Union #
  Gabon  |  Guyana  |  India  |  Indonesia
  Iran  |  Iraq  |  Italy  |  Kazakhstan
  Kuwait  |  Libya  |  Malaysia  |  Mexico
  Nigeria  |  Non-OECD  |  Non-OPEC  |  Norway
  OPEC  |  Oman  |  Other Africa  |  Other Asia Pacific
  Other CIS  |  Other Europe  |  Other Middle East  |  Other S. & Cent. America
  Peru  |  Qatar  |  Republic of Congo  |  Romania
  Russian Federation  |  Saudi Arabia  |  South Sudan  |  Sudan
  Syria  |  Thailand  |  Total Africa  |  Total Asia Pacific
  Total CIS  |  Total Europe  |  Total Middle East  |  Total North America
  Total S. & Cent. America  |  Total World  |  Trinidad & Tobago  |  Tunisia
  Turkmenistan  |  US  |  United Arab Emirates  |  United Kingdom
  Uzbekistan  |  Venezuela  |  Vietnam  |  Yemen
  of which: OECD